In [ ]:
import json
import requests
import pandas as pd

In [ ]:
def readSMILES(infile):
    
    smiles_list = []
    
    with open(infile) as f:
        f1 = f.readlines()
    
    for i in f1:
        smiles_list.append(i.strip())

In [ ]:
def divide_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [ ]:
def transform(data):
    resultList = []
    for mol in data['data']:
        if not mol['data']:
            # Invalid SMILES
            tmp = {'smiles': mol['smiles']}
        else:
            tmp = dict({'smiles': mol['smiles']})
            for _, admet in mol['data'].items():
                for endpoint in admet:
                    # endpoint is a dict
                    tmp[endpoint['name']] = endpoint['value']
        resultList.append(tmp)
    return pd.DataFrame(resultList).fillna('Invalid SMILES')

In [ ]:
baseUrl = 'https://admetlab3.scbdd.com'
api = '/api/admet'
url = baseUrl + api

param = {
        'SMILES': []
    }

n = 1000

infile = ""
smiles_list = readSMILES(infile)

for _, sublist in enumerate(divide_list(smiles_list, n)):
    
    param['SMILES'] = sublist

    response = requests.post(url, json=param)

    if response.status_code == 200:  # If access is successful
        data = response.json()['data']
        # transform to csv file 
        result = transform(data)
        result.to_csv('result.csv', index=False)